# OLMo-3 7B: raw next-token predictions (pretraining-only checkpoint)

Loads `allenai/Olmo-3-1025-7B` at revision **`stage1-step1413814`**, the final checkpoint of stage 1
pretraining (5.93T tokens). It comes **before** midtraining (`stage2`, which includes ~8% thinking and ~6%
instruction data), long-context training (`stage3`), and any SFT/DPO/RL. The `main` branch is the
model *after* stage 2 and stage 3.

This is a plain next-token predictor. There's no chat template, and **nothing is added to your input**:
no BOS token, no special tokens. Cells below assert this.

**This checkpoint differs from `main`:** the context window is 8192 tokens (no YaRN rope scaling).

**You need a GPU.** The login VM has none. See `README.md` for how to start Jupyter on `l40-worker`.

## Config

In [ ]:
import os

# Set before any Hugging Face import. Home is on /mnt/nw (shared with the GPU node), so the cache is too.
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
# Offline: load only what's already downloaded; never silently fetch or update anything.
# First-time download (run once in a terminal on the login VM):
#   ~/dev/verbalization/.venv/bin/hf download allenai/Olmo-3-1025-7B --revision stage1-step1413814
os.environ["HF_HUB_OFFLINE"] = "1"

MODEL_ID = "allenai/Olmo-3-1025-7B"
REVISION = "stage1-step1413814"
DEVICE = "cuda:0"
MAX_CTX = 8192          # this checkpoint was trained with 8k context
EOS_ID = 100257         # <|endoftext|>, the document separator in pretraining data
PAD_ID = 100277         # <|pad|>

## Environment

In [ ]:
import sys, shutil
import torch, transformers, tokenizers, huggingface_hub

print("python        ", sys.version.split()[0], sys.executable)
print("torch         ", torch.__version__)
print("transformers  ", transformers.__version__)
print("tokenizers    ", tokenizers.__version__)
print("hf_hub        ", huggingface_hub.__version__)
print("HF_HOME       ", os.environ["HF_HOME"])
print("free disk     ", f"{shutil.disk_usage(os.environ['HF_HOME']).free / 1e12:.1f} TB")

assert torch.cuda.is_available(), "No GPU visible: run this kernel on l40-worker via srun (see README.md)."
p = torch.cuda.get_device_properties(0)
print("gpu           ", p.name, f"{p.total_memory / 1e9:.0f} GB")

## Resolve the exact local snapshot
Everything below loads from this directory. The directory name is the commit SHA.

In [ ]:
from huggingface_hub import snapshot_download

SNAPSHOT = snapshot_download(MODEL_ID, revision=REVISION, local_files_only=True)
COMMIT = os.path.basename(SNAPSHOT)
print(SNAPSHOT)
print("revision", REVISION, "-> commit", COMMIT)

## Tokenizer
Text is encoded with the raw `tokenizers` library straight from the checkpoint's `tokenizer.json`, bypassing the
transformers wrapper. This checkpoint's `tokenizer_config.json` *defines* `bos_token = <|endoftext|>`, so we
check that the wrapper doesn't add it, either with `add_special_tokens=False` or with default flags.

In [ ]:
from tokenizers import Tokenizer
from transformers import AutoTokenizer

raw_tok = Tokenizer.from_file(os.path.join(SNAPSHOT, "tokenizer.json"))
tok = AutoTokenizer.from_pretrained(SNAPSHOT)   # used only for decode / id->token display

print("special tokens:")
for name in ["bos_token", "eos_token", "pad_token", "unk_token"]:
    t = getattr(tok, name)
    print(f"  {name:10} {t!r:18} id={tok.convert_tokens_to_ids(t) if t else None}")

tests = ["Hello world", " Hello", "Hello ", "\n\nThe capital of France is", "a<|endoftext|>b", ""]
for s in tests:
    raw = raw_tok.encode(s, add_special_tokens=False).ids
    assert raw == raw_tok.encode(s).ids, f"raw tokenizer adds tokens for {s!r}"
    assert tok.encode(s, add_special_tokens=False) == raw, f"wrapper differs for {s!r}"
    assert tok.encode(s) == raw, f"wrapper's default encode adds tokens for {s!r}"
    print(f"{s!r:32} -> {raw}")
print("OK: no BOS or other special tokens are added on encode.")

Note: a literal `<|endoftext|>` *inside your text* is parsed as the special token 100257 (see `'a<|endoftext|>b'` above).

## Load the model

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(SNAPSHOT, dtype=torch.bfloat16, device_map=DEVICE).eval()
torch.set_grad_enabled(False)

c = model.config
print("params                 ", f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f} B")
print("dtype                  ", model.dtype)
print("max_position_embeddings", c.max_position_embeddings)
print("rope_scaling           ", c.rope_scaling)
print("generation_config      ", model.generation_config.to_diff_dict())
print("gpu mem allocated      ", f"{torch.cuda.memory_allocated() / 1e9:.1f} GB")
assert c.max_position_embeddings == MAX_CTX

## Input / display helpers
These only build the input tensor or print things. All model computation below is a direct library call.

In [ ]:
def to_ids(x, prepend_eos=False):
    """str -> exact token ids (no special tokens); list[int] -> passed through untouched. Returns [1, T] tensor."""
    ids = raw_tok.encode(x, add_special_tokens=False).ids if isinstance(x, str) else list(x)
    if prepend_eos:
        ids = [EOS_ID] + ids
    assert 0 < len(ids) <= MAX_CTX, f"{len(ids)} tokens; must be 1..{MAX_CTX}"
    return torch.tensor([ids], device=model.device)

def show_tokens(input_ids):
    """Every input token: position, id, BPE token (Ġ = leading space, Ċ = newline), decoded text."""
    for i, t in enumerate(input_ids[0].tolist()):
        print(f"{i:>5} {t:>7}  {tok.convert_ids_to_tokens(t)!r:20} {tok.decode([t])!r}")

def show_topk(logprobs, k=20):
    top = torch.topk(logprobs, k)
    print(f"{'rank':>4} {'id':>7}  {'token':20} {'prob':>8} {'logprob':>9}")
    for r, (lp, i) in enumerate(zip(top.values.tolist(), top.indices.tolist())):
        print(f"{r:>4} {i:>7}  {tok.decode([i])!r:20} {torch.tensor(lp).exp().item():>8.4f} {lp:>9.3f}")

## Next-token distribution
One forward pass and the logits at the last position. Logits are cast to float32 before softmax.

In [ ]:
prompt = "The capital of France is"
input_ids = to_ids(prompt)
show_tokens(input_ids)

logits = model(input_ids).logits[0, -1].float()   # [vocab]
logprobs = logits.log_softmax(-1)
show_topk(logprobs, k=20)

You can also pass raw token IDs, or mark a document start with a leading `<|endoftext|>`:

In [ ]:
ids = to_ids(prompt).tolist()[0]
assert torch.equal(to_ids(ids), to_ids(prompt))            # raw ids pass through unchanged

show_topk(model(to_ids(prompt, prepend_eos=True)).logits[0, -1].float().log_softmax(-1), k=10)

## Continuations with `model.generate`
Every setting is passed explicitly. Unless overridden, transformers 5 fills in library defaults (`max_length=20`,
and **`top_k=50` when sampling**). The checkpoint's own `generation_config.json` only sets eos/pad IDs.
`output_logits=True` returns the *raw* model logits at each step (before any temperature or top-k processing),
and `compute_transition_scores` turns them into per-token log-probs.

In [ ]:
input_ids = to_ids("The capital of France is")
out = model.generate(
    input_ids,
    attention_mask=torch.ones_like(input_ids),
    max_new_tokens=30,
    do_sample=False,              # greedy
    eos_token_id=EOS_ID,          # stop at end-of-document; set to None to never stop early
    pad_token_id=PAD_ID,
    return_dict_in_generate=True,
    output_logits=True,
)
new = out.sequences[0, input_ids.shape[1]:]
lps = model.compute_transition_scores(out.sequences, out.logits, normalize_logits=True)[0]
print(repr(tok.decode(out.sequences[0])))
for t, lp in zip(new.tolist(), lps.tolist()):
    print(f"{t:>7}  {tok.decode([t])!r:20} {lp:>8.3f}")

In [ ]:
# Pure ancestral sampling from the full distribution: top_k=0 and top_p=1.0 disable truncation.
torch.manual_seed(0)
out = model.generate(
    input_ids,
    attention_mask=torch.ones_like(input_ids),
    max_new_tokens=40,
    do_sample=True,
    temperature=1.0,
    top_k=0,
    top_p=1.0,
    eos_token_id=EOS_ID,
    pad_token_id=PAD_ID,
)
print(repr(tok.decode(out[0])))

## Scoring a given continuation
One forward pass on `prefix + continuation`. The logits at position *i* predict token *i+1*.
Caveat: tokenizing prefix and continuation separately can split differently from tokenizing the joined string.
`show_tokens` lets you check.

In [ ]:
prefix = to_ids("The capital of France is")
cont = to_ids(" Paris, which is")
full = torch.cat([prefix, cont], dim=1)

lp = model(full).logits[0].float().log_softmax(-1)
T = prefix.shape[1]
cont_lps = lp[T - 1 : -1].gather(-1, cont[0][:, None])[:, 0]
for t, l in zip(cont[0].tolist(), cont_lps.tolist()):
    print(f"{t:>7}  {tok.decode([t])!r:20} {l:>8.3f}")
print("total logprob", cont_lps.sum().item())

## Sanity checks

In [ ]:
input_ids = to_ids("The capital of France is")
T = input_ids.shape[1]

# 1. Passing only input_ids == passing explicit all-ones mask and 0..T-1 positions (no hidden inputs differ).
a = model(input_ids).logits
b = model(input_ids, attention_mask=torch.ones_like(input_ids),
          position_ids=torch.arange(T, device=model.device)[None]).logits
assert torch.equal(a, b)

# 2. generate() prepends nothing and agrees with the direct forward pass.
out = model.generate(input_ids, attention_mask=torch.ones_like(input_ids), max_new_tokens=1,
                     do_sample=False, pad_token_id=PAD_ID)
assert torch.equal(out[:, :T], input_ids), "generate modified the prompt"
assert out[0, T].item() == a[0, -1].argmax().item()

# 3. Plausibility.
top10 = [tok.decode([i]) for i in a[0, -1].topk(10).indices.tolist()]
assert " Paris" in top10, top10
print("all sanity checks passed; top-10:", top10)

### Pitfall: trailing spaces
BPE attaches the space to the *next* word (`Ġ` = leading space), so a trailing space in your prompt changes the input and the prediction.

In [ ]:
for p in ["The capital of France is", "The capital of France is "]:
    ids = to_ids(p)
    print(repr(p), ids[0].tolist())
    show_topk(model(ids).logits[0, -1].float().log_softmax(-1), k=5)
    print()

## Scratch

In [ ]:
x = "Once upon a time"
show_topk(model(to_ids(x)).logits[0, -1].float().log_softmax(-1), k=20)